In [1]:
"""
March Machine Learning Mania 2026
==================================
DATA PREPROCESSING PIPELINE
==================================

Produces two clean output tables:
  - team_season_profiles.csv   : one row per (Season, TeamID) with all features
  - training_matchups.csv      : one row per past tournament game with feature diffs + label

Run:
    python preprocessing.py

Requires:  pip install pandas numpy scipy
All CSV files must be in DATA_DIR (default: current folder).
"""

import os
import numpy as np
import pandas as pd
from scipy.stats import mstats   # for winsorization

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────

DATA_DIR          = "C:\\Users\\eduah\\Desktop\\Kaggle Competions\\March ML Mania\\"                        # folder containing all CSVs
OUT_PROFILES      = "team_season_profiles_2.csv"  # Table A output
OUT_MATCHUPS      = "training_matchups_2.csv"     # Table B output

# Only use seasons from this year onward (older data = different era of basketball)
MIN_SEASON        = 2003   # 2003 is when detailed box scores begin for men
MIN_SEASON_W      = 2010   # 2010 is when detailed box scores begin for women

# A team must have played at least this many games to be included
MIN_GAMES         = 10

# Winsorize score margins at these percentile bounds (clips extreme blowouts)
WINSOR_LOW        = 0.01   # 1st percentile
WINSOR_HIGH       = 0.99   # 99th percentile

# Massey ranking systems to keep (well-established, available most years)
# Full list is huge; we cherry-pick the most trusted ones
MASSEY_SYSTEMS    = ["POM", "SAG", "MOR", "RPI", "KPI", "KEN", "DUN", "RTH"]

# Recency weighting: each season's stats are weighted by this factor
# Season 2025 weight = 1.0, Season 2024 = 0.85, Season 2023 = 0.72, etc.
RECENCY_DECAY     = 0.85

# ─────────────────────────────────────────────────────────────
# SECTION 1 — LOAD RAW DATA
# ─────────────────────────────────────────────────────────────

def load_all():
    """Load every CSV we need. Print a summary row count for each."""

    def read(fname):
        path = os.path.join(DATA_DIR, fname)
        df   = pd.read_csv(path)
        print(f"  ✓ {fname:<45s} {len(df):>8,} rows")
        return df

    print("\n" + "="*60)
    print("STEP 1 — Loading raw CSV files")
    print("="*60)

    d = {}

    # ── Identity ──────────────────────────────────────────────
    d["m_teams"]         = read("MTeams.csv")
    d["w_teams"]         = read("WTeams.csv")
    d["m_team_conf"]     = read("MTeamConferences.csv")
    d["w_team_conf"]     = read("WTeamConferences.csv")
    d["conferences"]     = read("Conferences.csv")
    d["m_coaches"]       = read("MTeamCoaches.csv")

    # ── Game results ──────────────────────────────────────────
    d["m_reg_detail"]    = read("MRegularSeasonDetailedResults.csv")
    d["w_reg_detail"]    = read("WRegularSeasonDetailedResults.csv")

    # ── Tournament ────────────────────────────────────────────
    d["m_tourney"]       = read("MNCAATourneyCompactResults.csv")
    d["w_tourney"]       = read("WNCAATourneyCompactResults.csv")
    d["m_seeds"]         = read("MNCAATourneySeeds.csv")
    d["w_seeds"]         = read("WNCAATourneySeeds.csv")

    # ── Rankings (men only) ───────────────────────────────────
    d["m_rankings"]      = read("MMasseyOrdinals.csv")

    # ── Submission template ───────────────────────────────────
    d["sample_sub"]      = read("SampleSubmissionStage2.csv")

    return d


# ─────────────────────────────────────────────────────────────
# SECTION 2 — BUILD MASTER TEAM TABLE
# Spine: one row per (Season, TeamID)
# ─────────────────────────────────────────────────────────────

def build_master_team_table(d):
    """
    Merge team identity, conference name, and end-of-season coach
    into a single lookup table.
    """
    print("\n" + "="*60)
    print("STEP 2 — Building master team identity table")
    print("="*60)

    # ── Men's identity ────────────────────────────────────────
    # MTeams has one row per team (no season column).
    # MTeamConferences has one row per (Season, TeamID).
    # We join on TeamID to attach team name to every season row.

    m_conf = d["m_team_conf"].copy()
    m_conf = m_conf.merge(d["m_teams"][["TeamID", "TeamName"]],
                          on="TeamID", how="left")
    m_conf = m_conf.merge(d["conferences"], on="ConfAbbrev", how="left")
    m_conf.rename(columns={"Description": "ConferenceName"}, inplace=True)
    m_conf["IsWomens"] = 0

    # ── Women's identity ──────────────────────────────────────
    w_conf = d["w_team_conf"].copy()
    w_conf = w_conf.merge(d["w_teams"][["TeamID", "TeamName"]],
                          on="TeamID", how="left")
    w_conf = w_conf.merge(d["conferences"], on="ConfAbbrev", how="left")
    w_conf.rename(columns={"Description": "ConferenceName"}, inplace=True)
    w_conf["IsWomens"] = 1

    # ── Combine ───────────────────────────────────────────────
    master = pd.concat([m_conf, w_conf], ignore_index=True)

    # ── Attach end-of-season coach (men only) ─────────────────
    # A team can have multiple coaches in one season (mid-season firing).
    # We keep the one with the highest LastDayNum (end of season).
    coaches = d["m_coaches"].copy()
    end_of_season_coach = (
        coaches.sort_values("LastDayNum", ascending=False)
               .groupby(["Season", "TeamID"])
               .first()
               .reset_index()[["Season", "TeamID", "CoachName"]]
    )
    master = master.merge(end_of_season_coach,
                          on=["Season", "TeamID"], how="left")
    master["CoachName"] = master["CoachName"].fillna("unknown")

    print(f"  ✓ Master team table: {len(master):,} rows "
          f"({master['IsWomens'].eq(0).sum():,} men's, "
          f"{master['IsWomens'].eq(1).sum():,} women's)")

    return master


# ─────────────────────────────────────────────────────────────
# SECTION 3 — COMPUTE REGULAR SEASON STATS PER TEAM PER SEASON
# ─────────────────────────────────────────────────────────────

def _flatten_game_rows(df):
    """
    Each game row has stats for BOTH the winning team and the losing team.
    We "flatten" it into two rows — one per team — so we can aggregate
    stats from each team's perspective regardless of outcome.

    This is the key structural transformation for all game data.
    """

    # ── Columns that exist in detailed results ────────────────
    stat_cols = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
                 "OR","DR","Ast","TO","Stl","Blk","PF"]

    # ── Winner perspective ────────────────────────────────────
    w_rename = {"WTeamID": "TeamID", "LTeamID": "OppID",
                "WScore": "Score",   "LScore": "OppScore"}
    for s in stat_cols:
        w_rename[f"W{s}"] = s
        w_rename[f"L{s}"] = f"Opp{s}"
    wins = df.rename(columns=w_rename).copy()
    wins["Win"]  = 1
    wins["Home"] = (wins["WLoc"] == "H").astype(int)

    # ── Loser perspective ─────────────────────────────────────
    l_rename = {"LTeamID": "TeamID", "WTeamID": "OppID",
                "LScore": "Score",   "WScore": "OppScore"}
    for s in stat_cols:
        l_rename[f"L{s}"] = s
        l_rename[f"W{s}"] = f"Opp{s}"
    losses = df.rename(columns=l_rename).copy()
    losses["Win"]  = 0
    losses["Home"] = (losses["WLoc"] == "A").astype(int)
    # WLoc = "A" means the WINNER was away → loser was home

    keep = (["Season", "DayNum", "TeamID", "OppID", "Score", "OppScore",
             "Win", "Home", "NumOT"] +
            stat_cols + [f"Opp{s}" for s in stat_cols])

    # Only keep columns present in both (compact results won't have stat_cols)
    keep = [c for c in keep if c in wins.columns and c in losses.columns]

    flat = pd.concat([wins[keep], losses[keep]], ignore_index=True)
    return flat


def _compute_possessions(row):
    """
    Estimate number of possessions using the standard formula.
    This is used to normalize counting stats per possession
    (removes the effect of pace — fast teams score more because
    they take more shots, not necessarily because they're better).

    Formula: Poss ≈ FGA - OR + TO + 0.44 * FTA
    """
    try:
        return row["FGA"] - row["OR"] + row["TO"] + 0.44 * row["FTA"]
    except Exception:
        return np.nan


def compute_team_stats(reg_detail_df, gender_label, min_season):
    """
    Aggregate regular season detailed results into per-team per-season stats.

    Preprocessing applied here:
      - Minimum games filter (MIN_GAMES)
      - Winsorization of score margins
      - Tempo normalization (per-possession rates)
      - Late-season form (last 30 days)
    """
    print(f"\n  → {gender_label}: filtering to Season >= {min_season}")
    df = reg_detail_df[reg_detail_df["Season"] >= min_season].copy()

    # ── Flatten to per-team rows ──────────────────────────────
    flat = _flatten_game_rows(df)

    stat_cols = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
                 "OR","DR","Ast","TO","Stl","Blk","PF"]

    # ── Derived raw stats ─────────────────────────────────────
    flat["Margin"]    = flat["Score"] - flat["OppScore"]
    flat["TotalReb"]  = flat["OR"] + flat["DR"]

    # ── Winsorize margins (clip extreme blowouts) ─────────────
    # We do this BEFORE aggregating so one 60-point blowout
    # doesn't distort a team's average margin
    flat["Margin"] = mstats.winsorize(
        flat["Margin"].values,
        limits=[WINSOR_LOW, 1 - WINSOR_HIGH]
    )

    # ── Possession estimate for tempo normalization ───────────
    flat["Poss"] = (flat["FGA"] - flat["OR"] + flat["TO"]
                    + 0.44 * flat["FTA"]).clip(lower=1)
    flat["OppPoss"] = (flat["OppFGA"] - flat["OppOR"] + flat["OppTO"]
                       + 0.44 * flat["OppFTA"]).clip(lower=1)

    # Points per 100 possessions (offensive & defensive efficiency)
    flat["OffRtg"] = (flat["Score"]    / flat["Poss"])    * 100
    flat["DefRtg"] = (flat["OppScore"] / flat["OppPoss"]) * 100
    flat["NetRtg"] = flat["OffRtg"] - flat["DefRtg"]

    # ── Full-season aggregation ───────────────────────────────
    agg = flat.groupby(["Season", "TeamID"]).agg(
        # Volume
        Games        = ("Win",      "count"),
        Wins         = ("Win",      "sum"),

        # Scoring
        AvgScore     = ("Score",    "mean"),
        AvgAllowed   = ("OppScore", "mean"),
        AvgMargin    = ("Margin",   "mean"),

        # Shooting (raw)
        FGM          = ("FGM",      "mean"),
        FGA          = ("FGA",      "mean"),
        FGM3         = ("FGM3",     "mean"),
        FGA3         = ("FGA3",     "mean"),
        FTM          = ("FTM",      "mean"),
        FTA          = ("FTA",      "mean"),

        # Opponent shooting
        OppFGM       = ("OppFGM",   "mean"),
        OppFGA       = ("OppFGA",   "mean"),

        # Misc
        AvgOR        = ("OR",       "mean"),
        AvgDR        = ("DR",       "mean"),
        AvgAst       = ("Ast",      "mean"),
        AvgTO        = ("TO",       "mean"),
        AvgStl       = ("Stl",      "mean"),
        AvgBlk       = ("Blk",      "mean"),
        AvgPF        = ("PF",       "mean"),

        # Efficiency (tempo-adjusted)
        AvgOffRtg    = ("OffRtg",   "mean"),
        AvgDefRtg    = ("DefRtg",   "mean"),
        AvgNetRtg    = ("NetRtg",   "mean"),

        # Home/Away context
        HomeGames    = ("Home",     "sum"),
    ).reset_index()

    # ── Derived shooting percentages ──────────────────────────
    # Safe division — avoid divide-by-zero
    agg["WinRate"]   = agg["Wins"]   / agg["Games"]
    agg["FGPct"]     = agg["FGM"]    / agg["FGA"].replace(0, np.nan)
    agg["FG3Pct"]    = agg["FGM3"]   / agg["FGA3"].replace(0, np.nan)
    agg["FTPct"]     = agg["FTM"]    / agg["FTA"].replace(0, np.nan)
    agg["OppFGPct"]  = agg["OppFGM"] / agg["OppFGA"].replace(0, np.nan)
    agg["AstTORatio"]= agg["AvgAst"] / agg["AvgTO"].replace(0, np.nan)

    # Two-point field goals specifically
    agg["FGM2"]      = agg["FGM"]  - agg["FGM3"]
    agg["FGA2"]      = agg["FGA"]  - agg["FGA3"]
    agg["FG2Pct"]    = agg["FGM2"] / agg["FGA2"].replace(0, np.nan)

    # ── Late-season form (last 30 days of regular season) ─────
    # DayNum goes up to 132 (Selection Sunday).
    # "Late season" = games on DayNum >= 102 (last ~30 days).
    late = flat[flat["DayNum"] >= 102].groupby(["Season", "TeamID"]).agg(
        LateGames      = ("Win",      "count"),
        LateWinRate    = ("Win",      "mean"),
        LateAvgMargin  = ("Margin",   "mean"),
        LateAvgNetRtg  = ("NetRtg",   "mean"),
    ).reset_index()

    agg = agg.merge(late, on=["Season", "TeamID"], how="left")
    # Teams with no late-season games (shouldn't happen) get NaN → fill below

    # ── Minimum games filter ──────────────────────────────────
    before = len(agg)
    agg = agg[agg["Games"] >= MIN_GAMES].copy()
    print(f"  → Removed {before - len(agg)} team-seasons with < {MIN_GAMES} games")

    # ── Fill remaining NaNs with column medians ───────────────
    # (affects late-season stats for teams with few late games,
    #  and shooting % for teams that never attempted certain shots)
    numeric_cols = agg.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if agg[col].isna().any():
            agg[col] = agg[col].fillna(agg[col].median())

    print(f"  ✓ {gender_label}: {len(agg):,} team-season rows")
    return agg


# ─────────────────────────────────────────────────────────────
# SECTION 4 — SEEDS
# ─────────────────────────────────────────────────────────────

def process_seeds(seeds_df, label):
    """
    Extract the numeric seed (1–16) from the seed string.
    e.g. "W03" → 3,  "X16b" → 16

    Teams NOT in the tournament get SeedNum = 17
    (worse than the worst possible seed, used as a signal
     that this team didn't qualify).
    """
    print(f"\n  → Processing {label} seeds")

    s = seeds_df.copy()
    s["SeedNum"] = s["Seed"].str[1:3].astype(int)

    # Keep only Season, TeamID, SeedNum
    s = s[["Season", "TeamID", "SeedNum"]]
    print(f"  ✓ {label}: {len(s):,} seeded team-seasons")
    return s


# ─────────────────────────────────────────────────────────────
# SECTION 5 — MASSEY RANKINGS (men only)
# ─────────────────────────────────────────────────────────────

def process_massey(rankings_df):
    """
    From 5.8M rows, extract only:
      - Final pre-tournament snapshot (RankingDayNum = 133)
      - A subset of trusted ranking systems

    Pivot to wide format: one row per (Season, TeamID),
    one column per ranking system.

    Missing ranks (team not ranked by a system) → filled with
    median rank for that season (≈ 175), meaning "average team".
    """
    print("\n  → Processing Massey ordinals (large file, please wait...)")

    # Filter to final pre-tournament rankings only
    final = rankings_df[rankings_df["RankingDayNum"] == 133].copy()

    # Filter to our chosen systems (keep only those that exist in the data)
    available = final["SystemName"].unique()
    systems   = [s for s in MASSEY_SYSTEMS if s in available]
    missing   = [s for s in MASSEY_SYSTEMS if s not in available]

    if missing:
        print(f"  ⚠ Systems not found in data: {missing}")

    print(f"  → Using {len(systems)} systems: {systems}")

    final = final[final["SystemName"].isin(systems)]

    # Pivot: rows = (Season, TeamID), columns = one per system
    pivoted = final.pivot_table(
        index   = ["Season", "TeamID"],
        columns = "SystemName",
        values  = "OrdinalRank",
        aggfunc = "mean"   # in case of duplicates
    ).reset_index()

    # Rename columns to avoid confusion
    pivoted.columns.name = None
    pivoted = pivoted.rename(
        columns={s: f"Rank_{s}" for s in systems}
    )

    rank_cols = [f"Rank_{s}" for s in systems]

    # Fill missing ranks with season median (not global median —
    # number of teams changes over years)
    for col in rank_cols:
        pivoted[col] = pivoted.groupby("Season")[col].transform(
            lambda x: x.fillna(x.median())
        )

    # AvgRank: mean across all available systems (lower = better)
    pivoted["AvgMasseyRank"] = pivoted[rank_cols].mean(axis=1)

    # Cap ranks at 351 (max D1 teams in any season) — data integrity
    for col in rank_cols + ["AvgMasseyRank"]:
        pivoted[col] = pivoted[col].clip(upper=351)

    print(f"  ✓ Massey rankings: {len(pivoted):,} team-seasons")
    return pivoted


# ─────────────────────────────────────────────────────────────
# SECTION 6 — ASSEMBLE TABLE A: TEAM SEASON PROFILES
# ─────────────────────────────────────────────────────────────

def build_team_profiles(master, m_stats, w_stats,
                        m_seeds_clean, w_seeds_clean,
                        massey):
    """
    Join everything into a single wide table:
      master (identity) + stats + seeds + rankings

    One row per (Season, TeamID).
    """
    print("\n" + "="*60)
    print("STEP 6 — Assembling Team Season Profiles (Table A)")
    print("="*60)

    # ── Combine men's and women's stats ───────────────────────
    all_stats = pd.concat([m_stats, w_stats], ignore_index=True)

    # ── Combine seeds ─────────────────────────────────────────
    all_seeds = pd.concat([m_seeds_clean, w_seeds_clean], ignore_index=True)

    # ── Start with master identity table ─────────────────────
    profiles = master.merge(all_stats, on=["Season", "TeamID"], how="inner")
    # inner join: only keep teams that have stats (filters out pre-2003 men's
    # and pre-2010 women's, per MIN_SEASON settings)

    # ── Add seeds ─────────────────────────────────────────────
    profiles = profiles.merge(all_seeds, on=["Season", "TeamID"], how="left")
    # Teams not in the tournament get NaN → fill with 17
    profiles["SeedNum"] = profiles["SeedNum"].fillna(17).astype(int)

    n_unseeded = (profiles["SeedNum"] == 17).sum()
    print(f"  ✓ {n_unseeded:,} team-seasons filled with SeedNum=17 (not in tournament)")

    # ── Add Massey rankings (men only) ────────────────────────
    profiles = profiles.merge(massey, on=["Season", "TeamID"], how="left")
    # Women's teams get NaN for Massey columns → fill with 200 (neutral)
    massey_cols = [c for c in profiles.columns if c.startswith("Rank_") or
                   c == "AvgMasseyRank"]
    for col in massey_cols:
        profiles[col] = profiles[col].fillna(200)

    # ── Add recency weight ────────────────────────────────────
    # Useful downstream when building training data
    max_season = profiles["Season"].max()
    profiles["RecencyWeight"] = RECENCY_DECAY ** (max_season - profiles["Season"])

    # ── Final check: no NaN in key columns ───────────────────
    key_cols = ["WinRate", "AvgMargin", "FGPct", "AvgOffRtg",
                "AvgNetRtg", "SeedNum"]
    for col in key_cols:
        n_nan = profiles[col].isna().sum()
        if n_nan > 0:
            median_val = profiles[col].median()
            profiles[col] = profiles[col].fillna(median_val)
            print(f"  ⚠ Filled {n_nan} NaNs in '{col}' with median={median_val:.3f}")

    print(f"\n  ✓ Final Table A: {len(profiles):,} rows × {len(profiles.columns)} columns")
    print(f"  ✓ Seasons covered: {profiles['Season'].min()} – {profiles['Season'].max()}")
    print(f"  ✓ Men's rows:   {profiles['IsWomens'].eq(0).sum():,}")
    print(f"  ✓ Women's rows: {profiles['IsWomens'].eq(1).sum():,}")

    return profiles


# ─────────────────────────────────────────────────────────────
# SECTION 7 — ASSEMBLE TABLE B: TRAINING MATCHUPS
# ─────────────────────────────────────────────────────────────

def build_training_matchups(m_tourney, w_tourney, profiles):
    """
    For each historical NCAA tournament game, build a feature row
    of DIFFERENCES between the two teams.

    Convention (consistent with submission format):
      Team1 = lower TeamID
      Team2 = higher TeamID
      Label = 1 if Team1 (lower ID) won, else 0

    Feature values = Team1_value − Team2_value
    Positive diff → Team1 is better on that stat.

    LEAKAGE PREVENTION:
      We only use pre-tournament stats (regular season aggregates).
      Tournament box scores (MNCAATourneyDetailedResults) are
      never used as features.
    """
    print("\n" + "="*60)
    print("STEP 7 — Building Training Matchups (Table B)")
    print("="*60)

    tourney = pd.concat([
        m_tourney.assign(IsWomens=0),
        w_tourney.assign(IsWomens=1)
    ], ignore_index=True)

    # Feature columns to difference (exclude identity + meta columns)
    exclude = {"Season", "TeamID", "TeamName", "ConfAbbrev", "ConferenceName",
               "CoachName", "IsWomens", "RecencyWeight", "Games", "Wins",
               "HomeGames", "LateGames"}
    feat_cols = [c for c in profiles.columns if c not in exclude]

    rows   = []
    skipped = 0

    for _, game in tourney.iterrows():
        season = game["Season"]
        t1     = min(game["WTeamID"], game["LTeamID"])
        t2     = max(game["WTeamID"], game["LTeamID"])
        label  = 1 if game["WTeamID"] == t1 else 0

        # Look up this season's profile for each team
        p1 = profiles[(profiles["Season"] == season) &
                      (profiles["TeamID"] == t1)]
        p2 = profiles[(profiles["Season"] == season) &
                      (profiles["TeamID"] == t2)]

        if p1.empty or p2.empty:
            skipped += 1
            continue

        p1 = p1.iloc[0]
        p2 = p2.iloc[0]

        row = {
            "Season":    season,
            "T1":        t1,
            "T2":        t2,
            "Label":     label,
            "IsWomens":  int(game["IsWomens"]),
            # Recency weight for this training example
            "RecencyWeight": p1["RecencyWeight"],
        }

        # Compute feature differences
        for col in feat_cols:
            try:
                row[f"{col}_diff"] = float(p1[col]) - float(p2[col])
            except (TypeError, ValueError):
                pass   # skip non-numeric columns silently

        rows.append(row)

    matchups = pd.DataFrame(rows)

    print(f"  ✓ Training matchups created: {len(matchups):,}")
    print(f"  ⚠ Games skipped (no profile data): {skipped}")

    # Label balance check — should be close to 50/50
    balance = matchups["Label"].mean()
    print(f"  ✓ Label balance: {balance:.3f} "
          f"(lower-ID team won {balance*100:.1f}% of games — "
          f"expected ~50%)")

    # Season distribution
    season_counts = matchups.groupby("Season")["Label"].count()
    print(f"  ✓ Season range: {season_counts.index.min()} – "
          f"{season_counts.index.max()}")

    return matchups


# ─────────────────────────────────────────────────────────────
# SECTION 8 — VALIDATE SUBMISSION COVERAGE
# ─────────────────────────────────────────────────────────────

def check_submission_coverage(sample_sub, profiles):
    """
    Check that every team mentioned in the submission file
    has a profile we can look up. Flag any gaps so we know
    what needs a fallback strategy.
    """
    print("\n" + "="*60)
    print("STEP 8 — Checking 2026 submission coverage")
    print("="*60)

    # Parse all unique TeamIDs needed for 2026 predictions
    sub_teams = set()
    for id_str in sample_sub["ID"]:
        parts = id_str.split("_")
        sub_teams.add(int(parts[1]))
        sub_teams.add(int(parts[2]))

    # Check which teams have 2025 or 2026 profile data
    recent = profiles[profiles["Season"] >= 2025]
    profiled_teams = set(recent["TeamID"].unique())

    covered   = sub_teams & profiled_teams
    uncovered = sub_teams - profiled_teams

    print(f"  ✓ Teams needed for submission: {len(sub_teams):,}")
    print(f"  ✓ Teams with recent profile:   {len(covered):,}")

    if uncovered:
        print(f"  ⚠ Teams WITHOUT recent profile: {len(uncovered)}")
        print(f"    These will fall back to historical averages at prediction time.")
        # Show a few examples
        print(f"    Example missing IDs: {sorted(list(uncovered))[:10]}")
    else:
        print(f"  ✓ All teams covered — no fallback needed!")

    return covered, uncovered


# ─────────────────────────────────────────────────────────────
# SECTION 9 — SAVE OUTPUTS
# ─────────────────────────────────────────────────────────────

def save_outputs(profiles, matchups):
    print("\n" + "="*60)
    print("STEP 9 — Saving output files")
    print("="*60)

    profiles.to_csv(OUT_PROFILES, index=False)
    print(f"  ✓ Saved Table A: {OUT_PROFILES}")
    print(f"    Shape: {profiles.shape[0]:,} rows × {profiles.shape[1]} columns")

    matchups.to_csv(OUT_MATCHUPS, index=False)
    print(f"  ✓ Saved Table B: {OUT_MATCHUPS}")
    print(f"    Shape: {matchups.shape[0]:,} rows × {matchups.shape[1]} columns")

    # Print column summary for both tables
    print(f"\n  Table A columns ({len(profiles.columns)}):")
    for i, col in enumerate(profiles.columns):
        dtype = profiles[col].dtype
        n_null = profiles[col].isna().sum()
        null_str = f" ← {n_null} NaN!" if n_null > 0 else ""
        print(f"    [{i:02d}] {col:<30s} {str(dtype):<12s}{null_str}")

    diff_cols = [c for c in matchups.columns if c.endswith("_diff")]
    meta_cols = [c for c in matchups.columns if not c.endswith("_diff")]
    print(f"\n  Table B: {len(meta_cols)} meta cols + {len(diff_cols)} diff features")
    print(f"  Meta: {meta_cols}")


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def main():
    print("\n" + "█"*60)
    print("  MARCH MADNESS 2026 — PREPROCESSING PIPELINE")
    print("█"*60)

    # 1. Load
    d = load_all()

    # 2. Master team identity
    master = build_master_team_table(d)

    # 3. Regular season stats
    print("\n" + "="*60)
    print("STEP 3 — Computing regular season stats")
    print("="*60)
    m_stats = compute_team_stats(d["m_reg_detail"], "Men's",   MIN_SEASON)
    w_stats = compute_team_stats(d["w_reg_detail"], "Women's", MIN_SEASON_W)

    # 4. Seeds
    print("\n" + "="*60)
    print("STEP 4 — Processing seeds")
    print("="*60)
    m_seeds_clean = process_seeds(d["m_seeds"], "Men's")
    w_seeds_clean = process_seeds(d["w_seeds"], "Women's")

    # 5. Massey rankings
    print("\n" + "="*60)
    print("STEP 5 — Processing Massey rankings")
    print("="*60)
    massey = process_massey(d["m_rankings"])

    # 6. Table A
    profiles = build_team_profiles(
        master, m_stats, w_stats,
        m_seeds_clean, w_seeds_clean,
        massey
    )

    # 7. Table B
    matchups = build_training_matchups(
        d["m_tourney"], d["w_tourney"], profiles
    )

    # 8. Submission coverage check
    check_submission_coverage(d["sample_sub"], profiles)

    # 9. Save
    save_outputs(profiles, matchups)

    print("\n" + "█"*60)
    print("  PREPROCESSING COMPLETE")
    print(f"  → {OUT_PROFILES}")
    print(f"  → {OUT_MATCHUPS}")
    print("█"*60 + "\n")


if __name__ == "__main__":
    main()


████████████████████████████████████████████████████████████
  MARCH MADNESS 2026 — PREPROCESSING PIPELINE
████████████████████████████████████████████████████████████

STEP 1 — Loading raw CSV files
  ✓ MTeams.csv                                         381 rows
  ✓ WTeams.csv                                         379 rows
  ✓ MTeamConferences.csv                            13,753 rows
  ✓ WTeamConferences.csv                             9,853 rows
  ✓ Conferences.csv                                     51 rows
  ✓ MTeamCoaches.csv                                13,900 rows
  ✓ MRegularSeasonDetailedResults.csv              124,529 rows
  ✓ WRegularSeasonDetailedResults.csv               87,187 rows
  ✓ MNCAATourneyCompactResults.csv                   2,585 rows
  ✓ WNCAATourneyCompactResults.csv                   1,717 rows
  ✓ MNCAATourneySeeds.csv                            2,694 rows
  ✓ WNCAATourneySeeds.csv                            1,812 rows
  ✓ MMasseyOrdinals.csv        

In [4]:
"""
Coverage Diagnostic
===================
Checks exactly which teams are in the submission but missing
from training matchups, and why.

Run: python diagnose_coverage.py
"""

OUT_DIR = "/kaggle/working/"
DATA_DIR = "/kaggle/input/competitions/march-machine-learning-mania-2026"
# Load the outputs from preprocessing
profiles = pd.read_csv(f"team_season_profiles_2.csv")
matchups = pd.read_csv(f"training_matchups_2.csv")
sample   = pd.read_csv("SampleSubmissionStage2.csv")
m_teams  = pd.read_csv("MTeams.csv")
w_teams  = pd.read_csv("WTeams.csv")
m_seeds  = pd.read_csv("MNCAATourneySeeds.csv")
w_seeds  = pd.read_csv("WNCAATourneySeeds.csv")

all_teams = pd.concat([
    m_teams[["TeamID","TeamName"]],
    w_teams[["TeamID","TeamName"]]
])

# ── 1. How many unique teams appear in training matchups? ──
t1s = set(matchups["T1"].unique())
t2s = set(matchups["T2"].unique())
in_training = t1s | t2s
print(f"\nUnique teams that appear in training matchups : {len(in_training)}")

# ── 2. How many unique teams are in the profiles? ──
in_profiles = set(profiles["TeamID"].unique())
print(f"Unique teams in team_season_profiles          : {len(in_profiles)}")

# ── 3. How many unique teams does the submission need? ──
sub_teams = set()
for id_str in sample["ID"]:
    parts = id_str.split("_")
    sub_teams.add(int(parts[1]))
    sub_teams.add(int(parts[2]))
print(f"Unique teams needed for submission            : {len(sub_teams)}")

# ── 4. Which submission teams have NO profile at all? ──
no_profile = sub_teams - in_profiles
print(f"\nSubmission teams with NO profile              : {len(no_profile)}")
if no_profile:
    missing_df = all_teams[all_teams["TeamID"].isin(no_profile)]
    print(missing_df.to_string(index=False))

# ── 5. Which submission teams have a profile but NEVER
#       appeared in any training matchup? ──
has_profile_not_trained = (sub_teams & in_profiles) - in_training
print(f"\nTeams with profile but never in training      : {len(has_profile_not_trained)}")
if has_profile_not_trained:
    df = all_teams[all_teams["TeamID"].isin(has_profile_not_trained)].copy()
    # Show their most recent season and seed
    recent = profiles[profiles["TeamID"].isin(has_profile_not_trained)]\
               .sort_values("Season",ascending=False)\
               .groupby("TeamID").first()[["Season","SeedNum","WinRate","AvgNetRtg"]]\
               .reset_index()
    df = df.merge(recent, on="TeamID", how="left")
    print(df.to_string(index=False))

# ── 6. Seed coverage: how many 2026 submission teams
#       have EVER been seeded in the tournament? ──
ever_seeded_m = set(m_seeds["TeamID"].unique())
ever_seeded_w = set(w_seeds["TeamID"].unique())
ever_seeded   = ever_seeded_m | ever_seeded_w

sub_mens   = {t for t in sub_teams if t < 3000}
sub_womens = {t for t in sub_teams if t >= 3000}

print(f"\nMen's submission teams   : {len(sub_mens)}")
print(f"  Ever seeded            : {len(sub_mens & ever_seeded_m)}")
print(f"  Never seeded           : {len(sub_mens - ever_seeded_m)}")

print(f"\nWomen's submission teams : {len(sub_womens)}")
print(f"  Ever seeded            : {len(sub_womens & ever_seeded_w)}")
print(f"  Never seeded           : {len(sub_womens - ever_seeded_w)}")

# ── 7. Profile season distribution ──
print("\nProfile season distribution (last 5 years):")
recent_profiles = profiles[profiles["Season"] >= 2021]
print(recent_profiles.groupby(["Season","IsWomens"])["TeamID"]\
      .count().rename("TeamCount").to_string())


Unique teams that appear in training matchups : 508
Unique teams in team_season_profiles          : 739
Unique teams needed for submission            : 728

Submission teams with NO profile              : 0

Teams with profile but never in training      : 224
 TeamID         TeamName  Season  SeedNum  WinRate  AvgNetRtg
   1108        Alcorn St    2026       17 0.281250 -17.986360
   1117      Arkansas St    2026       17 0.600000   3.935458
   1119             Army    2026       17 0.300000 -12.607974
   1123          Ball St    2026       17 0.344828  -9.396890
   1126  Bethune-Cookman    2026       17 0.516129  -1.903603
   1132    Bowling Green    2026       17 0.517241   5.413813
   1135            Brown    2026       17 0.280000  -8.970553
   1144         Campbell    2026       17 0.437500  -5.422566
   1145         Canisius    2026       17 0.300000 -14.135985
   1146    Cent Arkansas    2026       17 0.625000   3.863986
   1149    Charleston So    2026       17 0.413793   0.92

In [ ]:
# Compare old vs updated preprocessing outputs
import os
import pandas as pd

def compare_csv_versions(old_file, new_file, key_cols=None, sample_n=5):
    print("\n" + "=" * 80)
    print(f"Comparing: {old_file}  vs  {new_file}")
    print("=" * 80)

    if not os.path.exists(old_file):
        print(f"Missing file: {old_file}")
        return
    if not os.path.exists(new_file):
        print(f"Missing file: {new_file}")
        return

    old_df = pd.read_csv(old_file)
    new_df = pd.read_csv(new_file)

    # 1) Basic shape check
    print(f"Rows/Cols -> old: {old_df.shape}, new: {new_df.shape}")

    # 2) Column coverage check
    old_cols = set(old_df.columns)
    new_cols = set(new_df.columns)
    only_old = sorted(old_cols - new_cols)
    only_new = sorted(new_cols - old_cols)
    common = sorted(old_cols & new_cols)

    print(f"Common columns: {len(common)}")
    print(f"Only in old ({len(only_old)}): {only_old[:15]}{' ...' if len(only_old) > 15 else ''}")
    print(f"Only in new ({len(only_new)}): {only_new[:15]}{' ...' if len(only_new) > 15 else ''}")

    # 3) Dtype changes on common columns
    dtype_changes = []
    for c in common:
        dt_old = str(old_df[c].dtype)
        dt_new = str(new_df[c].dtype)
        if dt_old != dt_new:
            dtype_changes.append((c, dt_old, dt_new))

    print(f"Dtype changes: {len(dtype_changes)}")
    if dtype_changes:
        print("Sample dtype changes:")
        for c, dt_old, dt_new in dtype_changes[:15]:
            print(f"  - {c}: {dt_old} -> {dt_new}")

    # 4) Null count drift on common columns
    null_drift = []
    for c in common:
        old_null = int(old_df[c].isna().sum())
        new_null = int(new_df[c].isna().sum())
        if old_null != new_null:
            null_drift.append((c, old_null, new_null, new_null - old_null))

    print(f"Columns with null-count changes: {len(null_drift)}")
    if null_drift:
        print("Top null-count changes (by absolute delta):")
        for c, o, n, d in sorted(null_drift, key=lambda x: abs(x[3]), reverse=True)[:15]:
            print(f"  - {c}: {o} -> {n} (delta {d:+d})")

    # 5) Full equality check (only valid with identical columns/order/shape)
    if list(old_df.columns) == list(new_df.columns) and old_df.shape == new_df.shape:
        equal = old_df.equals(new_df)
        print(f"Exact equality (same shape/columns): {equal}")
    else:
        print("Exact equality: not comparable directly (shape/column mismatch)")

    # 6) Optional row-level diff by keys (useful for stable IDs)
    if key_cols is not None:
        missing_keys = [k for k in key_cols if k not in old_df.columns or k not in new_df.columns]
        if missing_keys:
            print(f"Key columns missing in one of the files: {missing_keys}")
        else:
            old_keys = set(map(tuple, old_df[key_cols].itertuples(index=False, name=None)))
            new_keys = set(map(tuple, new_df[key_cols].itertuples(index=False, name=None)))
            removed = old_keys - new_keys
            added = new_keys - old_keys

            print(f"Key rows removed: {len(removed)}")
            print(f"Key rows added:   {len(added)}")
            if removed:
                print(f"Sample removed keys: {list(removed)[:sample_n]}")
            if added:
                print(f"Sample added keys:   {list(added)[:sample_n]}")

# Pair 1: Team season profiles
compare_csv_versions(
    "team_season_profiles.csv",
    "team_season_profiles_2.csv",
    key_cols=["Season", "TeamID"]
)

# Pair 2: Training matchups
compare_csv_versions(
    "training_matchups.csv",
    "training_matchups_2.csv",
    key_cols=["Season", "T1", "T2"]
)